## Comparing dataset projections

In [38]:
import ee
import geemap
ee.Authenticate()
ee.Initialize()

# Map Initialization
Map = geemap.Map()

# Panama boundary
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")
panama_fc = countries.filter(ee.Filter.eq("ADM0_NAME", "Panama"))
panama_geom = panama_fc.geometry()

Map.centerObject(panama_geom, 7)

### Dynamic World LULC

In [39]:
START = "2024-01-01"
END = "2024-12-31"

# 10m resolution

dw_col = (
    ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
    .filterBounds(panama_geom)
    .filterDate(START, END)
)

dw_mode = dw_col.select("label").mode()

VIS_PALETTE = [
    "419bdf", "397d49", "88b053", "7a87c6",
    "e49635", "dfc35a", "c4281b", "a59b8f", "b39fe1"
]

Map.addLayer(
    dw_mode.clip(panama_geom),
    {"min": 0, "max": 8, "palette": VIS_PALETTE},
    "Dynamic World"
)

In [40]:
dw_mode.projection().getInfo()

{'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [1, 0, 0, 0, 1, 0]}

### Copernicus DEM (used in PRISM)

In [41]:
# DEM collection, ~30m resolution
dataset = (
    ee.ImageCollection('COPERNICUS/DEM/GLO30')
    .filterBounds(panama_geom)
)

# Create a single DEM image and clip it
elevation = (
    dataset
    .select('DEM')
    .mosaic()
    .clip(panama_geom)
)

elevation_vis = {
    'min': 0,
    'max': 2500,
    'palette': ['0000ff', '00ffff', 'ffff00', 'ff0000', 'ffffff'],
}

Map.addLayer(elevation, elevation_vis, 'Panama DEM')

In [42]:
elevation.projection().getInfo()

{'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [1, 0, 0, 0, 1, 0]}

### Distance to protected areas

In [43]:
pa_fc = ee.FeatureCollection("WCMC/WDPA/current/polygons").filterBounds(panama_geom)

pa_raster = ee.Image().byte().paint(pa_fc, 1)

dist_pa = (
    pa_raster.fastDistanceTransform(256)
    .sqrt()
    .multiply(30)
    .clip(panama_geom)
)

Map.addLayer(pa_fc, {}, "Protected Areas")
Map.addLayer(dist_pa, {}, "Distance Protected Areas")

In [46]:
# pa_fc.projection().getInfo()
dist_pa.projection().getInfo()

{'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [1, 0, 0, 0, 1, 0]}

### Generate Map

In [47]:
Map

Map(bottom=15906.0, center=[8.5158389458998, -80.10966640141521], controls=(WidgetControl(options=['position',…